In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")



In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018E2DA28050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018E2DA28C20>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage,AIMessage

model.invoke(
    [
        HumanMessage(content="Hi , My name is chandu and i am a AI Engineer"),
        AIMessage(content="Hello chandu its nice to meet u!"),
        HumanMessage(content="Hey What is my name and what do i do?")
    ]
)


AIMessage(content="Your name is Chandu, and you're an AI Engineer. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 80, 'total_tokens': 102, 'completion_time': 0.023497612, 'completion_tokens_details': None, 'prompt_time': 0.005021395, 'prompt_tokens_details': None, 'queue_time': 0.047958005, 'total_time': 0.028519007}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0ac-0a5e-73b3-a01d-4f758c434877-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 22, 'total_tokens': 102})

In [ ]:
#Message History

#We can use a Message History class to wrap our model and make it stateful.
#This will keep track of inputs and outputs of the model,and store them in some datastore

#Parent class for storing the messages
from langchain_core.chat_history import BaseChatMessageHistory

#It store the messages in the memory
from langchain_community.chat_message_histories import ChatMessageHistory


#It adds memory to a model/chain
from langchain_core.runnables.history import RunnableWithMessageHistory


store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()#It will create new object
    return store[session_id]

#Runnable is responsibe for handling memory chat
with_message_history=RunnableWithMessageHistory(model,get_session_history)


In [8]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is chandu and i am a AI Engineer")],
    config=config
)

In [14]:
with_message_history.invoke(

    [HumanMessage(content="What is my name")],
    config=config
)

    

AIMessage(content="Your name is Chandu. You're an AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 113, 'total_tokens': 126, 'completion_time': 0.019095707, 'completion_tokens_details': None, 'prompt_time': 0.006909135, 'prompt_tokens_details': None, 'queue_time': 0.047262255, 'total_time': 0.026004842}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0c6-8175-7640-9f73-a7fda7925fe1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_tokens': 13, 'total_tokens': 126})

In [15]:
store

{'chat1': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi , My name is chandu and i am a AI Engineer', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Chandu. As an AI Engineer, you must be working with various artificial intelligence technologies and developing innovative solutions. What specific areas of AI are you interested in, such as natural language processing, computer vision, or reinforcement learning?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 49, 'total_tokens': 100, 'completion_time': 0.076547655, 'completion_tokens_details': None, 'prompt_time': 0.003333225, 'prompt_tokens_details': None, 'queue_time': 0.045373565, 'total_time': 0.07988088}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0c4-f644-70c3-af3e-12d2f7c16cb4-0', to

In [ ]:
#Whenver when we invoke with_message_history model is passed and loaded at runtime 
#It pass the session id for memory

config2={"configurable":{"session_id":"chat2"}}
response2=with_message_history.invoke(
   [ HumanMessage(content="Im in hyderabad")],
    config=config2
)

In [22]:
response2

AIMessage(content="Hyderabad is a beautiful city with a rich history and culture. It's the capital of Telangana, a state in southern India. Here are a few suggestions for things to do or places to visit in Hyderabad:\n\n1. **Charminar**: A historic monument and one of the city's most iconic landmarks. It's a great place to take in the sights and sounds of old Hyderabad.\n2. **Golconda Fort**: A ancient fort that's steeped in history and offers breathtaking views of the city.\n3. **Hussain Sagar Lake**: A beautiful lake that's perfect for a relaxing stroll or a boat ride.\n4. **Laad Bazaar**: A bustling marketplace that's famous for its bangles, jewelry, and other handicrafts.\n5. **Hyderabad House**: A beautiful palace that was once the residence of the Nizam of Hyderabad.\n6. **Birla Mandir**: A stunning temple that's dedicated to Lord Venkateswara and offers beautiful views of the city.\n7. **KBR National Park**: A peaceful oasis in the heart of the city, perfect for a picnic or a wa

In [24]:
with_message_history.invoke(

    [HumanMessage(content="What is my city")],
    config=config2
)

    

AIMessage(content='Your city is Hyderabad.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 420, 'total_tokens': 426, 'completion_time': 0.014158504, 'completion_tokens_details': None, 'prompt_time': 0.025683511, 'prompt_tokens_details': None, 'queue_time': 0.046325198, 'total_time': 0.039842015}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0cd-d295-79d2-affc-ae02d3e72fc4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 420, 'output_tokens': 6, 'total_tokens': 426})

In [12]:
#Prompt Templates

#prompt Templates help to turn raw user information into a format that the LLm can work with.
#message placeholder is place where we pass all messages as a single variable

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are helpful assistant. Answer all the questions in this {language} "),
        MessagesPlaceholder(variable_name="messages")
    ]
)


In [54]:
chain=prompt|model

response=chain.invoke({"messages":[HumanMessage(content="Hello")],"language":"Telugu"})



response

AIMessage(content='నమస్కారం! నీకు ఏమి సహాయం చేయగలనో అడగండి.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 49, 'total_tokens': 118, 'completion_time': 0.103412086, 'completion_tokens_details': None, 'prompt_time': 0.002378257, 'prompt_tokens_details': None, 'queue_time': 0.051563103, 'total_time': 0.105790343}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf108-cef9-7ae0-aca5-d834d9ac0fa7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 69, 'total_tokens': 118})

In [27]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)


In [ ]:
config={"configurable":{"session_id":"chat"}}

response=with_message_history.invoke(
    [HumanMessage(content="Hi my name is chandu")],
    config=config
)

In [30]:
response

AIMessage(content='Nice to meet you, Chandu. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 51, 'total_tokens': 67, 'completion_time': 0.018106624, 'completion_tokens_details': None, 'prompt_time': 0.002942233, 'prompt_tokens_details': None, 'queue_time': 0.045549217, 'total_time': 0.021048857}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0f9-2a53-7613-8652-d7383046f78d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 16, 'total_tokens': 67})

In [31]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config
)
response

AIMessage(content='Your name is Chandu.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 80, 'total_tokens': 87, 'completion_time': 0.007134703, 'completion_tokens_details': None, 'prompt_time': 0.005331263, 'prompt_tokens_details': None, 'queue_time': 0.046784947, 'total_time': 0.012465966}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf0fa-9d77-7d70-bb05-8f3bd845e486-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 7, 'total_tokens': 87})

In [55]:
# With message key

with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [57]:
config={"configurable":{"session_id":"chat4"}}

response4=with_message_history.invoke(
    {"messages":[HumanMessage(content="hello")],"language":"Telugu"},
    config=config
)

In [58]:
response4

AIMessage(content='నమస్కారం. నాకు ఏమి తెలియజేయాలి?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 49, 'total_tokens': 104, 'completion_time': 0.093082298, 'completion_tokens_details': None, 'prompt_time': 0.002381847, 'prompt_tokens_details': None, 'queue_time': 0.073324363, 'total_time': 0.095464145}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf112-79e0-7293-aea2-7d69dc8ace40-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 55, 'total_tokens': 104})

In [13]:
#Managing the coversation history

#trim_messages helper to reduce how many messages we are sending to the model.The trimmer allows us to specify 
#how many tokens we want to keep

from langchain_core.messages import SystemMessage,trim_messages,HumanMessage,AIMessage

trimmer=trim_messages(
    max_tokens=40,
    strategy="last",#include last conversation
    token_counter=model,#it will count tokens
    include_system=True,#include system message
    allow_partial=False,
    start_on="human"
)

messages=[
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="Hi Im Bob"),
    AIMessage(content="Hi"),
    HumanMessage(content="2+2"),
    AIMessage(content="4")
]
trimmer.invoke(messages)

[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi Im Bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from operator import itemgetter
#Runnablepassthrough is used to pass same input to next step
from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model
)

In [17]:
response=chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What is my name")],
        "language":"English"
    }
)

In [18]:
response.content

'Your name is Bob.'